In [1]:
import pandas as pd
import numpy as np
import os
import gc  # Garbage collector
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')

In [2]:
import kagglehub

path = kagglehub.dataset_download("chethuhn/network-intrusion-dataset")

files = os.listdir(path)
csv_files = [f for f in files if f.endswith(".csv")]

print(f"Dataset path: {path}")
print(f"Found {len(csv_files)} CSV files")

Using Colab cache for faster access to the 'network-intrusion-dataset' dataset.
Dataset path: /kaggle/input/network-intrusion-dataset
Found 8 CSV files


In [3]:
import os

In [4]:
for file in csv_files:
    file_path = os.path.join(path, file)  # fix: define full path

    df = pd.read_csv(file_path, nrows=5)  # just read first 5 rows

    print(f"\nFile: {file}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(5))


File: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Shape: (5, 79)
Columns: [' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Variance', 'FIN Flag Count', ' S

In [42]:
# Step 2: Check NaN and infinite values
for file in csv_files:
    file_path = os.path.join(path, file)
    try:
        df = pd.read_csv(file_path)  # read sample for speed

        # Clean column names
        df.columns = df.columns.str.strip()

        # Count missing values
        missing_count = df.isnull().sum().sum()

        # Count infinite values
        inf_count = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()

        print(f"\nFile: {file}")
        print(f"  Shape: {df.shape}")
        print(f"  Total missing values: {missing_count}")
        print(f"  Total infinite values: {inf_count}")

        # Free memory
        del df
        gc.collect()

    except Exception as e:
        print(f"  Error processing {file}: {e}")



File: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  Shape: (286467, 79)
  Total missing values: 15
  Total infinite values: 727

File: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  Shape: (170366, 79)
  Total missing values: 20
  Total infinite values: 250

File: Tuesday-WorkingHours.pcap_ISCX.csv
  Shape: (445909, 79)
  Total missing values: 201
  Total infinite values: 327

File: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  Shape: (225745, 79)
  Total missing values: 4
  Total infinite values: 64

File: Monday-WorkingHours.pcap_ISCX.csv
  Shape: (529918, 79)
  Total missing values: 64
  Total infinite values: 810

File: Friday-WorkingHours-Morning.pcap_ISCX.csv
  Shape: (191033, 79)
  Total missing values: 28
  Total infinite values: 216

File: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  Shape: (288602, 79)
  Total missing values: 18
  Total infinite values: 396

File: Wednesday-workingHours.pcap_ISCX.csv
  Shape: (692703, 79)
  Total miss

In [43]:
# Step 3: Clean NaN and Infinite values
cleaned_data = {}  # store cleaned DataFrames (optional)

for file in csv_files:
    file_path = os.path.join(path, file)
    try:
        df = pd.read_csv(file_path)  # sample for speed

        # Clean column names
        df.columns = df.columns.str.strip()

        # Replace infinite values with NaN
        df.replace([np.inf, -np.inf], np.nan, inplace=True)

        # Fill NaN values with 0 (you can also use mean or median)
        df.fillna(0, inplace=True)

        # Optional: store cleaned DataFrame
        cleaned_data[file] = df

        # Safe numeric check
        numeric_df = df.select_dtypes(include=[np.number])

        print(f"\nFile: {file} cleaned successfully")
        print(f"  Shape: {df.shape}")
        print(f"  Any NaN left? {df.isnull().sum().sum()}")
        print(f"  Any Inf left? {np.isinf(numeric_df.to_numpy()).sum()}")

    except Exception as e:
        print(f"  Error cleaning {file}: {e}")


File: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv cleaned successfully
  Shape: (286467, 79)
  Any NaN left? 0
  Any Inf left? 0

File: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv cleaned successfully
  Shape: (170366, 79)
  Any NaN left? 0
  Any Inf left? 0

File: Tuesday-WorkingHours.pcap_ISCX.csv cleaned successfully
  Shape: (445909, 79)
  Any NaN left? 0
  Any Inf left? 0

File: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv cleaned successfully
  Shape: (225745, 79)
  Any NaN left? 0
  Any Inf left? 0

File: Monday-WorkingHours.pcap_ISCX.csv cleaned successfully
  Shape: (529918, 79)
  Any NaN left? 0
  Any Inf left? 0

File: Friday-WorkingHours-Morning.pcap_ISCX.csv cleaned successfully
  Shape: (191033, 79)
  Any NaN left? 0
  Any Inf left? 0

File: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv cleaned successfully
  Shape: (288602, 79)
  Any NaN left? 0
  Any Inf left? 0

File: Wednesday-workingHours.pcap_ISCX.csv cleaned successfully
  Sha

In [44]:
from sklearn.ensemble import RandomForestClassifier

# Store all feature importance results
feature_importance_list = []

for file, df in cleaned_data.items():
    print(f"\nAnalyzing feature importance for: {file}")

    try:
        # Prepare features and target
        X = df.select_dtypes(include=[np.number]).drop('Label', axis=1, errors='ignore')
        y = df['Label'] if 'Label' in df.columns else None

        if y is not None and len(X) > 0:
            # Ensure target has no NaN
            y = y.fillna(0)

            # Train lightweight Random Forest
            rf = RandomForestClassifier(
                n_estimators=20,   # small number of trees
                max_depth=5,       # shallow trees for speed
                n_jobs=-1,         # use all CPU cores
                random_state=42
            )

            rf.fit(X, y)

            # Store feature importance
            for feat, imp in zip(X.columns, rf.feature_importances_):
                feature_importance_list.append({
                    'file': file,
                    'feature': feat,
                    'importance': imp
                })

        # Free memory
        del X, y, rf
        gc.collect()

    except Exception as e:
        print(f"  Error processing {file}: {e}")


Analyzing feature importance for: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv

Analyzing feature importance for: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv

Analyzing feature importance for: Tuesday-WorkingHours.pcap_ISCX.csv

Analyzing feature importance for: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv

Analyzing feature importance for: Monday-WorkingHours.pcap_ISCX.csv

Analyzing feature importance for: Friday-WorkingHours-Morning.pcap_ISCX.csv

Analyzing feature importance for: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv

Analyzing feature importance for: Wednesday-workingHours.pcap_ISCX.csv


In [45]:
# Convert list of feature importances to DataFrame
importance_df = pd.DataFrame(feature_importance_list)

# Check if DataFrame is not empty
if not importance_df.empty:
    # For each file, select top 20 features
    top20_features_per_file = (
        importance_df.groupby('file')
        .apply(lambda x: x.nlargest(20, 'importance'))
        .reset_index(drop=True)
    )

    # Display results
    for file in top20_features_per_file['file'].unique():
        print(f"\nTop 20 Features for {file}:")
        temp_df = top20_features_per_file[top20_features_per_file['file'] == file]
        for idx, row in temp_df.iterrows():
            print(f"{row['feature']}: {row['importance']:.4f}")
else:
    print("No feature importance data available.")


Top 20 Features for Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv:
Fwd Packet Length Max: 0.1210
Init_Win_bytes_forward: 0.0822
Subflow Fwd Packets: 0.0807
Fwd Packet Length Mean: 0.0804
Destination Port: 0.0714
Subflow Fwd Bytes: 0.0650
Avg Fwd Segment Size: 0.0641
Total Length of Fwd Packets: 0.0511
act_data_pkt_fwd: 0.0423
Fwd IAT Std: 0.0356
Fwd IAT Total: 0.0348
Bwd Packet Length Max: 0.0269
Bwd Packet Length Min: 0.0230
Fwd Header Length.1: 0.0229
Fwd IAT Mean: 0.0229
Fwd Packet Length Std: 0.0220
Avg Bwd Segment Size: 0.0211
Total Length of Bwd Packets: 0.0208
Fwd IAT Max: 0.0198
Flow IAT Std: 0.0163

Top 20 Features for Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv:
Fwd Packet Length Max: 0.1264
Flow Duration: 0.1083
Total Length of Fwd Packets: 0.0877
Subflow Fwd Bytes: 0.0842
Packet Length Mean: 0.0764
Fwd Packet Length Mean: 0.0566
Avg Fwd Segment Size: 0.0487
Avg Bwd Segment Size: 0.0413
Bwd Packets/s: 0.0393
Flow IAT Max: 0.0350
Subflow Fwd Packets: 0.0318
Fwd I

In [46]:
print(importance_df)


                                                  file  \
0    Friday-WorkingHours-Afternoon-PortScan.pcap_IS...   
1    Friday-WorkingHours-Afternoon-PortScan.pcap_IS...   
2    Friday-WorkingHours-Afternoon-PortScan.pcap_IS...   
3    Friday-WorkingHours-Afternoon-PortScan.pcap_IS...   
4    Friday-WorkingHours-Afternoon-PortScan.pcap_IS...   
..                                                 ...   
619               Wednesday-workingHours.pcap_ISCX.csv   
620               Wednesday-workingHours.pcap_ISCX.csv   
621               Wednesday-workingHours.pcap_ISCX.csv   
622               Wednesday-workingHours.pcap_ISCX.csv   
623               Wednesday-workingHours.pcap_ISCX.csv   

                         feature  importance  
0               Destination Port    0.000233  
1                  Flow Duration    0.108297  
2              Total Fwd Packets    0.010561  
3         Total Backward Packets    0.000699  
4    Total Length of Fwd Packets    0.087749  
..                   

In [47]:
# ============================================
# STEP 1: First, analyze feature importance using samples from each file
# ============================================
print("\n=== STEP 1: Analyzing Feature Importance ===")

# Use previously computed feature_importance_list
if feature_importance_list:
    # Convert to DataFrame
    importance_df = pd.DataFrame(feature_importance_list)

    # Average importance across all files
    importance_df = (
        importance_df.groupby('feature')['importance']
        .mean()
        .reset_index()
        .sort_values('importance', ascending=False)
        .reset_index(drop=True)
    )

    print("\nTop 20 Most Important Features (across all files):")
    print(importance_df.head(20).to_string(index=False))

    # Select top 30 features
    top_features = importance_df.head(30)['feature'].tolist()
    print(f"\nSelected {len(top_features)} top features for final model training")

else:
    # Fallback if feature importance is empty
    top_features = None
    print("Could not determine feature importance")


=== STEP 1: Analyzing Feature Importance ===

Top 20 Most Important Features (across all files):
                    feature  importance
           Destination Port    0.054324
      Fwd Packet Length Max    0.043238
          Subflow Fwd Bytes    0.039830
         Packet Length Mean    0.035151
Total Length of Fwd Packets    0.033885
       Avg Fwd Segment Size    0.033477
              Flow Duration    0.027546
       Avg Bwd Segment Size    0.027376
     Fwd Packet Length Mean    0.027162
    Init_Win_bytes_backward    0.025146
     Bwd Packet Length Mean    0.023620
          Packet Length Std    0.023561
          Max Packet Length    0.022849
Total Length of Bwd Packets    0.020954
              Fwd Packets/s    0.020878
          Fwd Header Length    0.020260
     Init_Win_bytes_forward    0.020002
     Packet Length Variance    0.019943
        Subflow Fwd Packets    0.019227
                Fwd IAT Min    0.017496

Selected 30 top features for final model training


In [48]:
# ============================================
# STEP 2: Processing Files with Selected Features (Shuffle + 70-80% from each file)
# ============================================

import random

all_data = []  # This will store processed dataframes
target_rows = 1800000  # Target total rows
current_rows = 0
frac_per_file = 0.85  # Use 85% of each file (adjust as needed, 0.85)

for file in csv_files:
    print(f"\nProcessing {file}...")
    file_path = os.path.join(path, file)

    try:
        # Read entire file (memory permitting)
        df = pd.read_csv(file_path)

        # Clean column names
        df.columns = df.columns.str.strip()

        # Shuffle rows
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)

        # Take fraction of data
        n_rows_to_take = int(len(df) * frac_per_file)
        df = df.iloc[:n_rows_to_take]

        # Keep only top features + Label
        if top_features:
            available_features = [f for f in top_features if f in df.columns]
            if 'Label' in df.columns:
                available_features.append('Label')
            df = df[available_features]

        # Clean data
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        df.dropna(inplace=True)

        # Downcast numeric types to save memory
        for col in df.select_dtypes(include=['int64']).columns:
            df[col] = pd.to_numeric(df[col], downcast='integer')
        for col in df.select_dtypes(include=['float64']).columns:
            df[col] = pd.to_numeric(df[col], downcast='float')

        # Add to list if not empty
        if not df.empty:
            all_data.append(df)
            current_rows += len(df)
            print(f"  Added {len(df)} rows from {file}, total rows so far: {current_rows}")

        # Stop if reached target
        if current_rows >= target_rows:
            print(f"\nReached target row count ({target_rows}). Stopping.")
            break

        # Free memory
        del df
        gc.collect()

    except Exception as e:
        print(f"  Error processing {file}: {e}")
        continue

# Combine all selected chunks into final_df
if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    print(f"\nFinal dataset shape after concatenation: {final_df.shape}")
else:
    print("No data processed successfully!")


Processing Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv...
  Added 243175 rows from Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv, total rows so far: 243175

Processing Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv...
  Added 144695 rows from Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv, total rows so far: 387870

Processing Tuesday-WorkingHours.pcap_ISCX.csv...
  Added 378793 rows from Tuesday-WorkingHours.pcap_ISCX.csv, total rows so far: 766663

Processing Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv...
  Added 191856 rows from Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv, total rows so far: 958519

Processing Monday-WorkingHours.pcap_ISCX.csv...
  Added 450067 rows from Monday-WorkingHours.pcap_ISCX.csv, total rows so far: 1408586

Processing Friday-WorkingHours-Morning.pcap_ISCX.csv...
  Added 162274 rows from Friday-WorkingHours-Morning.pcap_ISCX.csv, total rows so far: 1570860

Processing Thursday-WorkingHours-Afternoon-Infilteration.pc

In [49]:
print("\n=== STEP 4: Training Initial Model ===")

if 'final_df' in locals() and len(final_df) > 0 and 'Label' in final_df.columns:
    # Prepare data
    X = final_df.drop('Label', axis=1)
    y = final_df['Label']

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    # Train initial model
    rf_initial = RandomForestClassifier(
        n_estimators=50,
        max_depth=10,
        min_samples_split=50,
        min_samples_leaf=25,
        n_jobs=-1,
        random_state=42
    )

    print("Training initial model...")
    rf_initial.fit(X_train, y_train)

    # Evaluate initial model
    y_pred_initial = rf_initial.predict(X_test)
    print("\nInitial Model Classification Report:")
    print(classification_report(y_test, y_pred_initial))

    # Calculate initial accuracy
    initial_accuracy = accuracy_score(y_test, y_pred_initial)
    print(f"\n{'='*50}")
    print(f"INITIAL MODEL ACCURACY: {initial_accuracy:.6f} ({initial_accuracy*100:.4f}%)")
    print(f"{'='*50}")

else:
    print("Final dataset not available or missing 'Label' column.")


=== STEP 4: Training Initial Model ===
Training initial model...

Initial Model Classification Report:
                            precision    recall  f1-score   support

                    BENIGN       1.00      1.00      1.00    467037
                       Bot       1.00      0.34      0.50       499
                      DDoS       1.00      1.00      1.00     32651
               FTP-Patator       1.00      1.00      1.00      2025
              Infiltration       0.00      0.00      0.00        10
                  PortScan       0.99      1.00      1.00     40488
               SSH-Patator       1.00      0.98      0.99      1524
  Web Attack � Brute Force       0.69      0.86      0.77       393
Web Attack � Sql Injection       0.00      0.00      0.00         5
          Web Attack � XSS       0.00      0.00      0.00       165

                  accuracy                           1.00    544797
                 macro avg       0.67      0.62      0.63    544797
          

In [50]:
best_accuracy = initial_accuracy
best_params = {}
best_rf = rf_initial  # Start with initial model as best
results = []

In [15]:
# ============================================
# STEP 5: Sequential Parameter Testing (Alternative to RandomizedSearchCV)
# ============================================
print("\n" + "="*60)
print("STEP 5: Sequential Parameter Testing")
print("="*60)

# Define parameter combinations to test manually
param_combinations = [
    # Previously tested combinations (commented out for reference)
    # {'n_estimators': 100, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 2},
    # {'n_estimators': 100, 'max_depth': 30, 'min_samples_split': 10, 'min_samples_leaf': 4},
    # {'n_estimators': 150, 'max_depth': 25, 'min_samples_split': 8, 'min_samples_leaf': 3},
    # {'n_estimators': 200, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 2},
    # {'n_estimators': 200, 'max_depth': 30, 'min_samples_split': 10, 'min_samples_leaf': 4},
    # {'n_estimators': 250, 'max_depth': 25, 'min_samples_split': 8, 'min_samples_leaf': 3},
    # {'n_estimators': 300, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 2},
    # {'n_estimators': 300, 'max_depth': 30, 'min_samples_split': 10, 'min_samples_leaf': 4},

    # Best parameters from previous hyperparameter tuning
    {'n_estimators': 350, 'max_depth': 25, 'min_samples_split': 8, 'min_samples_leaf': 3},

    # Future combinations can be added here
    # {'n_estimators': 400, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 2},
]


STEP 5: Sequential Parameter Testing


In [16]:
print(f"\nTesting {len(param_combinations)} parameter combination(s) sequentially...")
print("This may take 15-30 minutes depending on your system...")

for i, params in enumerate(param_combinations, 1):
    print(f"\n{i}/{len(param_combinations)}. Testing: {params}")

    # Train model with these parameters
    rf_test = RandomForestClassifier(
        **params,
        n_jobs=-1,
        random_state=42,
        verbose=0
    )

    try:
        # Train model
        rf_test.fit(X_train, y_train)

        # Evaluate
        y_pred_test = rf_test.predict(X_test)
        acc_test = accuracy_score(y_test, y_pred_test)

        # Store results
        results.append({**params, 'accuracy': acc_test})
        print(f"   ✓ Accuracy: {acc_test:.6f} ({acc_test*100:.4f}%)")

        # Update best model if improved
        if acc_test > best_accuracy:
            prev_best = best_accuracy
            best_accuracy = acc_test
            best_params = params
            best_rf = rf_test
            print(f"   ⭐ NEW BEST MODEL! (Previous best: {prev_best:.6f})")

        # Free memory
        del rf_test
        gc.collect()

    except Exception as e:
        print(f"   ❌ Error during testing: {e}")
        continue


Testing 1 parameter combination(s) sequentially...
This may take 15-30 minutes depending on your system...

1/1. Testing: {'n_estimators': 350, 'max_depth': 25, 'min_samples_split': 8, 'min_samples_leaf': 3}


KeyboardInterrupt: 

In [51]:
# Show results
print("\n" + "="*60)
print("SEQUENTIAL TESTING RESULTS")
print("="*60)


SEQUENTIAL TESTING RESULTS


In [52]:
# # Create results dataframe
# if results:
#     results_df = pd.DataFrame(results).sort_values('accuracy', ascending=False)

#     print("\nTOP 5 PARAMETER COMBINATIONS:")
#     print(results_df.head(5).to_string(index=False))

#     print("\nBOTTOM 5 PARAMETER COMBINATIONS:")
#     print(results_df.tail(5).to_string(index=False))

#     # Summary statistics
#     print("\nAccuracy Statistics:")
#     print(f"  Mean: {results_df['accuracy'].mean():.6f}")
#     print(f"  Std : {results_df['accuracy'].std():.6f}")
#     print(f"  Min : {results_df['accuracy'].min():.6f}")
#     print(f"  Max : {results_df['accuracy'].max():.6f}")

# else:
#     print("No results to display!")

In [53]:
# ============================================
# BEST MODEL DETAILS (Using Initial Model)
# ============================================
print("\n" + "="*50)
print("BEST MODEL USED (Initial Model)")
print("="*50)

print("Model parameters used:")
for param, value in rf_initial.get_params().items():
    print(f"  {param}: {value}")

# Accuracy
print(f"\nModel accuracy (from initial evaluation): {initial_accuracy:.6f} ({initial_accuracy*100:.4f}%)")


BEST MODEL USED (Initial Model)
Model parameters used:
  bootstrap: True
  ccp_alpha: 0.0
  class_weight: None
  criterion: gini
  max_depth: 10
  max_features: sqrt
  max_leaf_nodes: None
  max_samples: None
  min_impurity_decrease: 0.0
  min_samples_leaf: 25
  min_samples_split: 50
  min_weight_fraction_leaf: 0.0
  monotonic_cst: None
  n_estimators: 50
  n_jobs: -1
  oob_score: False
  random_state: 42
  verbose: 0
  warm_start: False

Model accuracy (from initial evaluation): 0.998199 (99.8199%)


In [54]:
# # Compare with original model
# improvement = (best_accuracy - initial_accuracy) * 100
# print(f"\nInitial model accuracy: {initial_accuracy:.6f} ({initial_accuracy*100:.4f}%)")
# print(f"Best model accuracy: {best_accuracy:.6f} ({best_accuracy*100:.4f}%)")
# print(f"Improvement: +{improvement:.4f}%")

In [55]:
# # Show detailed classification report for best model
# print("\n" + "="*50)
# print("DETAILED CLASSIFICATION REPORT - BEST MODEL")
# print("="*50)
# y_pred_best = best_rf.predict(X_test)
# print(classification_report(y_test, y_pred_best))

In [56]:
# # Feature importance from best model
# print("\n" + "="*50)
# print("TOP 20 FEATURES - BEST MODEL")
# print("="*50)
# feature_importance_best = pd.DataFrame({
#     'feature': X.columns,
#     'importance': best_rf.feature_importances_
# }).sort_values('importance', ascending=False)

# for i, row in feature_importance_best.head(20).iterrows():
#     print(f"{i+1:2d}. {row['feature']:30s}: {row['importance']:.6f}")

In [57]:
# ============================================
# FEATURE IMPORTANCE - INITIAL MODEL
# ============================================
print("\n" + "="*50)
print("TOP 20 FEATURES - INITIAL MODEL")
print("="*50)

# Feature importance from initial model
feature_importance_initial = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_initial.feature_importances_
}).sort_values('importance', ascending=False)

# Display top 20 features
for i, row in feature_importance_initial.head(20).iterrows():
    print(f"{i+1:2d}. {row['feature']:30s}: {row['importance']:.6f}")


TOP 20 FEATURES - INITIAL MODEL
 8. Avg Bwd Segment Size          : 0.097519
18. Packet Length Variance        : 0.083019
12. Packet Length Std             : 0.076649
11. Bwd Packet Length Mean        : 0.070238
 5. Total Length of Fwd Packets   : 0.061079
30. Average Packet Size           : 0.057095
 4. Packet Length Mean            : 0.053622
 3. Subflow Fwd Bytes             : 0.041543
27. Bwd Packet Length Max         : 0.041227
 2. Fwd Packet Length Max         : 0.040267
22. Bwd Packets/s                 : 0.036398
 9. Fwd Packet Length Mean        : 0.033886
24. Subflow Bwd Bytes             : 0.032356
26. act_data_pkt_fwd              : 0.031606
 6. Avg Fwd Segment Size          : 0.027017
 1. Destination Port              : 0.024292
21. Bwd Header Length             : 0.022115
16. Fwd Header Length             : 0.022065
13. Max Packet Length             : 0.018808
19. Subflow Fwd Packets           : 0.018367


In [58]:
# print("\n" + "="*60)
# print("FINAL SUMMARY")
# print("="*60)
# print(f"Total samples processed: {len(final_df)}")
# print(f"Number of features used: {X.shape[1]}")
# print(f"Training set size: {len(X_train)}")
# print(f"Test set size: {len(X_test)}")
# print(f"Number of classes: {len(np.unique(y))}")
# print(f"\nInitial Model Accuracy: {initial_accuracy:.6f} ({initial_accuracy*100:.4f}%)")
# print(f"Best Model Accuracy: {best_accuracy:.6f} ({best_accuracy*100:.4f}%)")
# print(f"Total Improvement: +{improvement:.4f}%")


In [59]:
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)

print(f"Total samples processed: {len(final_df)}")
print(f"Number of features used: {X.shape[1]}")
print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Number of classes: {len(np.unique(y))}")
print(f"\nInitial Model Accuracy: {initial_accuracy:.6f} ({initial_accuracy*100:.4f}%)")


FINAL SUMMARY
Total samples processed: 1815989
Number of features used: 30
Training set size: 1271192
Test set size: 544797
Number of classes: 10

Initial Model Accuracy: 0.998199 (99.8199%)


In [60]:
# initial_errors = len(y_test) * (1 - initial_accuracy)
# best_errors = len(y_test) * (1 - best_accuracy)
# errors_saved = initial_errors - best_errors
# print(f"\nErrors with initial model: {initial_errors:.0f}")
# print(f"Errors with best model: {best_errors:.0f}")
# print(f"Errors saved: {errors_saved:.0f}")
# print(f"Error reduction: {(errors_saved/initial_errors)*100:.2f}%")

In [61]:
# print("\n" + "="*60)
# print("SHORT TEST ON UNSEEN TEST DATA")
# print("="*60)

# # Your model with your parameters
# rf_test_short = RandomForestClassifier(
#     n_estimators=30,
#     max_depth=10,
#     min_samples_leaf=20,
#     min_samples_split=30,
#     max_features="sqrt",
#     n_jobs=-1,
#     random_state=42
# )

In [62]:
print("\n" + "="*60)
print("SHORT TEST ON UNSEEN TEST DATA")
print("="*60)

# Small Random Forest for quick evaluation
rf_test_short = RandomForestClassifier(
    n_estimators=30,
    max_depth=10,
    min_samples_leaf=20,
    min_samples_split=30,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

print("Training short-test Random Forest on training data...")
rf_test_short.fit(X_train, y_train)

# Predict on test data
y_pred_short = rf_test_short.predict(X_test)

# Evaluate accuracy
short_accuracy = accuracy_score(y_test, y_pred_short)
print(f"\nShort test accuracy: {short_accuracy:.6f} ({short_accuracy*100:.4f}%)")

# Optional: detailed classification report
print("\nClassification report for short test model:")
print(classification_report(y_test, y_pred_short))


SHORT TEST ON UNSEEN TEST DATA
Training short-test Random Forest on training data...

Short test accuracy: 0.998221 (99.8221%)

Classification report for short test model:
                            precision    recall  f1-score   support

                    BENIGN       1.00      1.00      1.00    467037
                       Bot       1.00      0.35      0.52       499
                      DDoS       1.00      1.00      1.00     32651
               FTP-Patator       1.00      0.99      1.00      2025
              Infiltration       0.00      0.00      0.00        10
                  PortScan       0.99      1.00      1.00     40488
               SSH-Patator       1.00      0.97      0.99      1524
  Web Attack � Brute Force       0.69      0.86      0.77       393
Web Attack � Sql Injection       0.00      0.00      0.00         5
          Web Attack � XSS       0.00      0.00      0.00       165

                  accuracy                           1.00    544797
         

In [63]:
# ============================================
# Calculate total rows across all CSV files
# ============================================

total_rows_all_files = 0

for file in csv_files:
    file_path = os.path.join(path, file)
    try:
        # Read just first 5 rows to check columns and preview
        df_preview = pd.read_csv(file_path, nrows=5)
        print(f"\nFile: {file}")
        print(f"Preview shape: {df_preview.shape}")
        print(f"Columns: {df_preview.columns.tolist()}")

        # Count total rows in the file efficiently
        row_count = sum(1 for _ in open(file_path)) - 1  # subtract 1 for header
        print(f"Total rows in this file: {row_count}")

        total_rows_all_files += row_count
        del df_preview
        gc.collect()

    except Exception as e:
        print(f"  ❌ Could not process {file}: {e}")
        continue

print(f"\nTotal rows across all CSV files: {total_rows_all_files}")



File: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Preview shape: (5, 79)
Columns: [' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Variance', 'FIN Flag Cou

In [64]:
# ============================================
# Calculate percentage of total dataset used for model (considering cleaning)
# ============================================

rows_used_for_model = len(final_df)  # final_df contains all rows used (500k)

# Assume 200k rows were duplicates/NaNs and not actually used
cleaned_rows = total_rows_all_files - 200000  # ground reality rows

# Percentage of total dataset used (after cleaning)
percent_of_total_data_used = (rows_used_for_model / cleaned_rows) * 100

print(f"\nRows used for this model (train + test): {rows_used_for_model}")
print("Assuming out of total rows 200k are duplicates/NaNs, subtracting this to get the ground reality.")
print(f"Total rows across all CSV files: {total_rows_all_files}")
print(f"Rows after cleaning: {cleaned_rows}")
print(f"Percentage of total dataset used for this model (after cleaning): {percent_of_total_data_used:.2f}%")



Rows used for this model (train + test): 1815989
Assuming out of total rows 200k are duplicates/NaNs, subtracting this to get the ground reality.
Total rows across all CSV files: 2830743
Rows after cleaning: 2630743
Percentage of total dataset used for this model (after cleaning): 69.03%


In [65]:
# ============================================
# Display exact columns used for this model
# ============================================

if 'final_df' in locals() and len(final_df) > 0:

    # Columns used for the model
    model_columns = final_df.columns.tolist()

    print("\nColumns used for this model:")
    for i, col in enumerate(model_columns, 1):
        print(f"{i}. {col}")

else:
    print("Final dataset is empty or not available!")



Columns used for this model:
1. Destination Port
2. Fwd Packet Length Max
3. Subflow Fwd Bytes
4. Packet Length Mean
5. Total Length of Fwd Packets
6. Avg Fwd Segment Size
7. Flow Duration
8. Avg Bwd Segment Size
9. Fwd Packet Length Mean
10. Init_Win_bytes_backward
11. Bwd Packet Length Mean
12. Packet Length Std
13. Max Packet Length
14. Total Length of Bwd Packets
15. Fwd Packets/s
16. Fwd Header Length
17. Init_Win_bytes_forward
18. Packet Length Variance
19. Subflow Fwd Packets
20. Fwd IAT Min
21. Bwd Header Length
22. Bwd Packets/s
23. Flow Packets/s
24. Subflow Bwd Bytes
25. Fwd IAT Mean
26. act_data_pkt_fwd
27. Bwd Packet Length Max
28. Fwd IAT Total
29. Fwd Header Length.1
30. Average Packet Size
31. Label


In [66]:


# import pandas as pd

# if 'best_rf' in locals() and len(final_df) > 0:

#     # Get feature columns (exclude label)
#     feature_columns = [col for col in final_df.columns if col != 'Label']

#     print("\n=== User Input Prediction ===")
#     print("Please enter values for the following features:")

#     # Prepare empty dict to store user input
#     user_input_dict = {}

#     # Loop through each feature to take input
#     for feature in feature_columns:
#         while True:
#             try:
#                 val = float(input(f"Enter value for '{feature}': "))
#                 user_input_dict[feature] = val
#                 break
#             except ValueError:
#                 print("Invalid input. Please enter a numeric value.")

#     # Convert input to dataframe
#     user_input_df = pd.DataFrame([user_input_dict])

#     # Predict label using best model
#     predicted_label = best_rf.predict(user_input_df)[0]

#     print(f"\n✅ Predicted Label/Class: {predicted_label}")

# else:
#     print("Model or dataset not available. Train the model first!")


In [67]:
# We have used just 2.5 files out of 8 and by files name it seems like that each file has majority of one kind attacks
# Leading 5.5 files unused may lead to faulty result
#Result maybe biased
#Lets verify using the .valuecount .nunique .unique function on label colomn on label colomn on each file
#After that verify it using the data from files which we not used for this model


In [68]:
#Dummy Testing done Predicted wrong It is data of DOS but class predicted benign

In [69]:
# 0

# import pandas as pd

# if 'best_rf' in locals() and len(final_df) > 0:

#     # Get feature columns (exclude label)
#     feature_columns = [col for col in final_df.columns if col != 'Label']

#     print("\n=== User Input Prediction ===")
#     print("Please enter values for the following features:")

#     # Prepare empty dict to store user input
#     user_input_dict = {}

#     # Loop through each feature to take input
#     for feature in feature_columns:
#         while True:
#             try:
#                 val = float(input(f"Enter value for '{feature}': "))
#                 user_input_dict[feature] = val
#                 break
#             except ValueError:
#                 print("Invalid input. Please enter a numeric value.")

#     # Convert input to dataframe
#     user_input_df = pd.DataFrame([user_input_dict])

#     # Predict label using best model
#     predicted_label = best_rf.predict(user_input_df)[0]

#     print(f"\n✅ Predicted Label/Class: {predicted_label}")

# else:
#     print("Model or dataset not available. Train the model first!")


In [70]:
#it again predicted wrong it is web attack it classify as normal

In [71]:
#last verify on its dataset row

In [72]:
# # ============================================
# # Extract 1 row of 'Web Attack – XSS' from final_df
# # ============================================

# if 'final_df' in locals() and len(final_df) > 0:

#     # Define feature columns (exclude 'Label')
#     feature_columns = [col for col in final_df.columns if col != 'Label']

#     # Filter rows where label is 'Web Attack – XSS'
#     xss_rows = final_df[final_df['Label'].str.contains("Web Attack", na=False) &
#                         final_df['Label'].str.contains("XSS", na=False)]

#     if len(xss_rows) >= 1:
#         # Take first row
#         xss_row_sample = xss_rows.iloc[0][feature_columns]  # Series of features only
#         print("=== Extracted 1 row (features only) for Web Attack – XSS ===\n")

#         # Print in vertical format
#         for col, val in xss_row_sample.items():
#             print(f"{col}: {val}")

#     else:
#         print(f"No rows found for 'Web Attack – XSS'.")
# else:
#     print("final_df not available or empty!")


In [73]:
# 0

# import pandas as pd

# if 'best_rf' in locals() and len(final_df) > 0:

#     # Get feature columns (exclude label)
#     feature_columns = [col for col in final_df.columns if col != 'Label']

#     print("\n=== User Input Prediction ===")
#     print("Please enter values for the following features:")

#     # Prepare empty dict to store user input
#     user_input_dict = {}

#     # Loop through each feature to take input
#     for feature in feature_columns:
#         while True:
#             try:
#                 val = float(input(f"Enter value for '{feature}': "))
#                 user_input_dict[feature] = val
#                 break
#             except ValueError:
#                 print("Invalid input. Please enter a numeric value.")

#     # Convert input to dataframe
#     user_input_df = pd.DataFrame([user_input_dict])

#     # Predict label using best model
#     predicted_label = best_rf.predict(user_input_df)[0]

#     print(f"\n✅ Predicted Label/Class: {predicted_label}")

# else:
#     print("Model or dataset not available. Train the model first!")


In [74]:
# only predicted true on seen data


In [75]:
# We have used just 2.5 files out of 8 and by files name it seems like that each file has majority of one kind attacks
# Leading 5.5 files unused may lead to faulty result
#Result maybe biased
#Lets verify using the .valuecount .nunique .unique function on label colomn on label colomn on each file
#if each file has majority of any class then it means that we have to cover data from all files and increase the model rows for training

In [76]:
import pandas as pd
import os

# List of 5 files to check
files_to_check = [
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv"
]

# Loop through each file
for file in files_to_check:
    file_path = os.path.join(path, file)

    try:
        df = pd.read_csv(file_path, nrows=100000)  # read first 100k for speed

        # Strip column names just in case
        df.columns = df.columns.str.strip()

        if 'Label' in df.columns:
            print(f"\nFile: {file}")
            print(f"Number of unique labels (nunique): {df['Label'].nunique()}")
            print(f"Labels present (unique): {df['Label'].unique()}")
            print("Label distribution (value counts):")
            print(df['Label'].value_counts())
        else:
            print(f"\nFile: {file} has no 'Label' column after stripping spaces.")

    except Exception as e:
        print(f"Error reading {file}: {e}")



File: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Number of unique labels (nunique): 2
Labels present (unique): ['BENIGN' 'DDoS']
Label distribution (value counts):
Label
DDoS      61194
BENIGN    38806
Name: count, dtype: int64

File: Monday-WorkingHours.pcap_ISCX.csv
Number of unique labels (nunique): 1
Labels present (unique): ['BENIGN']
Label distribution (value counts):
Label
BENIGN    100000
Name: count, dtype: int64

File: Friday-WorkingHours-Morning.pcap_ISCX.csv
Number of unique labels (nunique): 2
Labels present (unique): ['BENIGN' 'Bot']
Label distribution (value counts):
Label
BENIGN    99558
Bot         442
Name: count, dtype: int64

File: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Number of unique labels (nunique): 2
Labels present (unique): ['BENIGN' 'Infiltration']
Label distribution (value counts):
Label
BENIGN          99982
Infiltration       18
Name: count, dtype: int64

File: Wednesday-workingHours.pcap_ISCX.csv
Number of unique labels (nuniq

In [85]:
# Ensure 'Label' column exists
if 'Label' in final_df.columns:
    # Filter rows where Label is 'DDoS'
    dos_rows = final_df[final_df['Label'] == 'DDoS']

    if not dos_rows.empty:
        # Take the first row
        dos_sample = dos_rows.iloc[0]  # as Series

        print("Extracted row where Label = 'DDoS' (vertical view):\n")
        for feature, value in dos_sample.items():
            print(f"{feature}: {value}")
    else:
        print("No row with Label = 'DDoS' found in the dataset.")
else:
    print("Label column not found in final_df.")

Extracted row where Label = 'DDoS' (vertical view):

Destination Port: 80
Fwd Packet Length Max: 6
Subflow Fwd Bytes: 24
Packet Length Mean: 6.0
Total Length of Fwd Packets: 24
Avg Fwd Segment Size: 6.0
Flow Duration: 9392887
Avg Bwd Segment Size: 0.0
Fwd Packet Length Mean: 6.0
Init_Win_bytes_backward: -1
Bwd Packet Length Mean: 0.0
Packet Length Std: 0.0
Max Packet Length: 6
Total Length of Bwd Packets: 0
Fwd Packets/s: 0.42585416
Fwd Header Length: 80
Init_Win_bytes_forward: 256
Packet Length Variance: 0.0
Subflow Fwd Packets: 4
Fwd IAT Min: 975
Bwd Header Length: 0
Bwd Packets/s: 0.0
Flow Packets/s: 0.42585416
Subflow Bwd Bytes: 0
Fwd IAT Mean: 3130962.333
act_data_pkt_fwd: 3
Bwd Packet Length Max: 0
Fwd IAT Total: 9392887
Fwd Header Length.1: 80
Average Packet Size: 7.5
Label: DDoS


In [86]:
import pandas as pd
import numpy as np

# ===============================
# Input row data for DDoS sample
# All 30 features included, Label is ignored for prediction
# ===============================
input_row_ddos = {
    'Destination Port': 80,
    'Fwd Packet Length Max': 6,
    'Subflow Fwd Bytes': 24,
    'Packet Length Mean': 6.0,
    'Total Length of Fwd Packets': 24,
    'Avg Fwd Segment Size': 6.0,
    'Flow Duration': 9392887,
    'Avg Bwd Segment Size': 0.0,
    'Fwd Packet Length Mean': 6.0,
    'Init_Win_bytes_backward': -1,
    'Bwd Packet Length Mean': 0.0,
    'Packet Length Std': 0.0,
    'Max Packet Length': 6,
    'Total Length of Bwd Packets': 0,
    'Fwd Packets/s': 0.42585416,
    'Fwd Header Length': 80,
    'Init_Win_bytes_forward': 256,
    'Packet Length Variance': 0.0,
    'Subflow Fwd Packets': 4,
    'Fwd IAT Min': 975,
    'Bwd Header Length': 0,
    'Bwd Packets/s': 0.0,
    'Flow Packets/s': 0.42585416,
    'Subflow Bwd Bytes': 0,
    'Fwd IAT Mean': 3130962.333,
    'act_data_pkt_fwd': 3,
    'Bwd Packet Length Max': 0,
    'Fwd IAT Total': 9392887,
    'Fwd Header Length.1': 80,
    'Average Packet Size': 7.5
}

# Convert to DataFrame
input_df_ddos = pd.DataFrame([input_row_ddos])

# Ensure columns are in the same order as model training features
model_features = X.columns.tolist()  # list of 30 top features used for training
input_df_ddos = input_df_ddos.reindex(columns=model_features, fill_value=0)

# ===============================
# Verification
# ===============================
missing_features = set(model_features) - set(input_df_ddos.columns)
extra_features = set(input_df_ddos.columns) - set(model_features)

if missing_features:
    print(f"❌ Missing features: {missing_features}")
if extra_features:
    print(f"⚠ Extra features: {extra_features}")
if not missing_features:
    print("✅ Input pipeline has all 30 features. Ready for prediction.")

# ===============================
# Predict using initial model
# ===============================
prediction = rf_initial.predict(input_df_ddos)
prediction_proba = rf_initial.predict_proba(input_df_ddos) if hasattr(rf_initial, "predict_proba") else None

print(f"\nPrediction for the DDoS row: {prediction[0]}")
if prediction_proba is not None:
    print(f"Prediction probabilities: {prediction_proba[0]}")

✅ Input pipeline has all 30 features. Ready for prediction.

Prediction for the DDoS row: DDoS
Prediction probabilities: [4.09472157e-04 0.00000000e+00 9.99590528e-01 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00]


In [87]:
import pandas as pd
import numpy as np

# ===============================
# Duplicate DDoS row (exact same values)
# ===============================
input_row_ddos_2 = {
    'Destination Port': 80,
    'Fwd Packet Length Max': 6,
    'Subflow Fwd Bytes': 24,
    'Packet Length Mean': 6.0,
    'Total Length of Fwd Packets': 24,
    'Avg Fwd Segment Size': 6.0,
    'Flow Duration': 9392887,
    'Avg Bwd Segment Size': 0.0,
    'Fwd Packet Length Mean': 6.0,
    'Init_Win_bytes_backward': -1,
    'Bwd Packet Length Mean': 0.0,
    'Packet Length Std': 0.0,
    'Max Packet Length': 6,
    'Total Length of Bwd Packets': 0,
    'Fwd Packets/s': 0.42585416,
    'Fwd Header Length': 80,
    'Init_Win_bytes_forward': 256,
    'Packet Length Variance': 0.0,
    'Subflow Fwd Packets': 4,
    'Fwd IAT Min': 975,
    'Bwd Header Length': 0,
    'Bwd Packets/s': 0.0,
    'Flow Packets/s': 0.42585416,
    'Subflow Bwd Bytes': 0,
    'Fwd IAT Mean': 3130962.333,
    'act_data_pkt_fwd': 3,
    'Bwd Packet Length Max': 0,
    'Fwd IAT Total': 9392887,
    'Fwd Header Length.1': 80,
    'Average Packet Size': 7.5
}

# Convert to DataFrame
input_df_ddos_2 = pd.DataFrame([input_row_ddos_2])

# Ensure correct feature order for the model
model_features = X.columns.tolist()
input_df_ddos_2 = input_df_ddos_2.reindex(columns=model_features, fill_value=0)

# ===============================
# Verification
# ===============================
missing_features = set(model_features) - set(input_df_ddos_2.columns)
extra_features = set(input_df_ddos_2.columns) - set(model_features)

if missing_features:
    print(f"❌ Missing features: {missing_features}")
if extra_features:
    print(f"⚠ Extra features: {extra_features}")
if not missing_features:
    print("✅ Duplicate DDoS input pipeline is correct. Ready for prediction.")

# ===============================
# Predict using initial model
# ===============================
prediction = rf_initial.predict(input_df_ddos_2)
prediction_proba = rf_initial.predict_proba(input_df_ddos_2) if hasattr(rf_initial, "predict_proba") else None

print(f"\nPrediction for the duplicate DDoS row: {prediction[0]}")
if prediction_proba is not None:
    print(f"Prediction probabilities: {prediction_proba[0]}")

✅ Duplicate DDoS input pipeline is correct. Ready for prediction.

Prediction for the duplicate DDoS row: DDoS
Prediction probabilities: [4.09472157e-04 0.00000000e+00 9.99590528e-01 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00]


In [88]:
import pandas as pd
import numpy as np

# ===============================
# Input data for 3 scenarios
# All 30 features included
# Label is ignored for prediction
# ===============================

# 1️⃣ Normal traffic (example values)
normal_row = {
    'Destination Port': 443,
    'Fwd Packet Length Max': 1500,
    'Subflow Fwd Bytes': 1200,
    'Packet Length Mean': 1000.0,
    'Total Length of Fwd Packets': 5000,
    'Avg Fwd Segment Size': 800.0,
    'Flow Duration': 500000,
    'Avg Bwd Segment Size': 1000.0,
    'Fwd Packet Length Mean': 950.0,
    'Init_Win_bytes_backward': 512,
    'Bwd Packet Length Mean': 980.0,
    'Packet Length Std': 50.0,
    'Max Packet Length': 1500,
    'Total Length of Bwd Packets': 4500,
    'Fwd Packets/s': 50.0,
    'Fwd Header Length': 60,
    'Init_Win_bytes_forward': 512,
    'Packet Length Variance': 2500.0,
    'Subflow Fwd Packets': 10,
    'Fwd IAT Min': 100,
    'Bwd Header Length': 60,
    'Bwd Packets/s': 45.0,
    'Flow Packets/s': 48.0,
    'Subflow Bwd Bytes': 4200,
    'Fwd IAT Mean': 1000.0,
    'act_data_pkt_fwd': 10,
    'Bwd Packet Length Max': 1200,
    'Fwd IAT Total': 10000,
    'Fwd Header Length.1': 60,
    'Average Packet Size': 1024.0
}

# 2️⃣ Suspicious traffic (example values)
suspicious_row = {
    'Destination Port': 8080,
    'Fwd Packet Length Max': 200,
    'Subflow Fwd Bytes': 800,
    'Packet Length Mean': 180.0,
    'Total Length of Fwd Packets': 1500,
    'Avg Fwd Segment Size': 170.0,
    'Flow Duration': 1000000,
    'Avg Bwd Segment Size': 150.0,
    'Fwd Packet Length Mean': 160.0,
    'Init_Win_bytes_backward': 256,
    'Bwd Packet Length Mean': 155.0,
    'Packet Length Std': 10.0,
    'Max Packet Length': 200,
    'Total Length of Bwd Packets': 1400,
    'Fwd Packets/s': 20.0,
    'Fwd Header Length': 60,
    'Init_Win_bytes_forward': 256,
    'Packet Length Variance': 100.0,
    'Subflow Fwd Packets': 8,
    'Fwd IAT Min': 200,
    'Bwd Header Length': 60,
    'Bwd Packets/s': 18.0,
    'Flow Packets/s': 19.0,
    'Subflow Bwd Bytes': 1300,
    'Fwd IAT Mean': 500.0,
    'act_data_pkt_fwd': 5,
    'Bwd Packet Length Max': 190,
    'Fwd IAT Total': 4000,
    'Fwd Header Length.1': 60,
    'Average Packet Size': 185.0
}

# 3️⃣ DDoS attack (your provided row)
ddos_row = {
    'Destination Port': 80,
    'Fwd Packet Length Max': 6,
    'Subflow Fwd Bytes': 24,
    'Packet Length Mean': 6.0,
    'Total Length of Fwd Packets': 24,
    'Avg Fwd Segment Size': 6.0,
    'Flow Duration': 9392887,
    'Avg Bwd Segment Size': 0.0,
    'Fwd Packet Length Mean': 6.0,
    'Init_Win_bytes_backward': -1,
    'Bwd Packet Length Mean': 0.0,
    'Packet Length Std': 0.0,
    'Max Packet Length': 6,
    'Total Length of Bwd Packets': 0,
    'Fwd Packets/s': 0.42585416,
    'Fwd Header Length': 80,
    'Init_Win_bytes_forward': 256,
    'Packet Length Variance': 0.0,
    'Subflow Fwd Packets': 4,
    'Fwd IAT Min': 975,
    'Bwd Header Length': 0,
    'Bwd Packets/s': 0.0,
    'Flow Packets/s': 0.42585416,
    'Subflow Bwd Bytes': 0,
    'Fwd IAT Mean': 3130962.333,
    'act_data_pkt_fwd': 3,
    'Bwd Packet Length Max': 0,
    'Fwd IAT Total': 9392887,
    'Fwd Header Length.1': 80,
    'Average Packet Size': 7.5
}

# ===============================
# Combine into single DataFrame
# ===============================
all_inputs = pd.DataFrame([normal_row, suspicious_row, ddos_row])

# Ensure correct column order
model_features = X.columns.tolist()  # your top 30 features
all_inputs = all_inputs.reindex(columns=model_features, fill_value=0)

# ===============================
# Verification
# ===============================
missing_features = set(model_features) - set(all_inputs.columns)
extra_features = set(all_inputs.columns) - set(model_features)

if missing_features:
    print(f"❌ Missing features: {missing_features}")
if extra_features:
    print(f"⚠ Extra features: {extra_features}")
if not missing_features:
    print("✅ All 3 input pipelines are correct. Ready for prediction.")

# ===============================
# Predict using initial model
# ===============================
predictions = rf_initial.predict(all_inputs)
predictions_proba = rf_initial.predict_proba(all_inputs) if hasattr(rf_initial, "predict_proba") else None

print("\nPredictions for all 3 scenarios:")
for i, pred in enumerate(predictions, 1):
    print(f"  Sample {i}: {pred}")

if predictions_proba is not None:
    print("\nPrediction probabilities for all 3 samples:")
    for i, probs in enumerate(predictions_proba, 1):
        print(f"  Sample {i}: {probs}")

✅ All 3 input pipelines are correct. Ready for prediction.

Predictions for all 3 scenarios:
  Sample 1: BENIGN
  Sample 2: BENIGN
  Sample 3: DDoS

Prediction probabilities for all 3 samples:
  Sample 1: [9.65437908e-01 1.48645831e-02 3.40068519e-06 0.00000000e+00
 6.89276190e-06 1.16827620e-05 1.95949830e-02 4.95022159e-05
 2.17519717e-05 9.29519071e-06]
  Sample 2: [8.67904653e-01 1.16635006e-01 2.01324284e-06 1.52173913e-02
 3.89820107e-05 9.98107124e-05 1.91913030e-05 4.53404709e-05
 2.51639217e-05 1.24479558e-05]
  Sample 3: [4.09472157e-04 0.00000000e+00 9.99590528e-01 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00]


In [96]:
df['Label'].unique()

array([1])